# Risk landscapes and cuts

Reproduce the main landscape (`set_1`) or an appendix landscape (`set_2`, `set_6`). Select `CASE` and run the notebook from top to bottom. The default loads the included numerical arrays, redraws the figure, and saves its PDF to `output/`. Set `RECOMPUTE=True` to run the theoretical searches and synthetic simulations from the explicit inputs.

The numerical risk $R$ excludes additive independent test-noise variance $\sigma^2$; the figures show $R/R_\infty$. The main case uses test differing-site mean 0.2 and 300 training trials. The appendix cases use theoretical curves only.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from copy import deepcopy
import sys
import importlib
import inspect
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# This notebook accepts only this bundle or its notebooks/ directory.
# It never discovers a parent research checkout or imports its cache.
candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
BUNDLE_ROOT = next((p for p in candidates
    if (p / 'dms').is_dir() and (p / 'notebooks' / '01_landscapes.ipynb').is_file()), None)
if BUNDLE_ROOT is None:
    raise RuntimeError('Start Jupyter in the reproducibility bundle or its notebooks/ directory.')
if 'dms' in sys.modules:
    loaded = Path(sys.modules['dms'].__file__).resolve().parent
    if loaded != BUNDLE_ROOT / 'dms':
        raise RuntimeError('Restart the kernel: dms was imported from another checkout.')
sys.path.insert(0, str(BUNDLE_ROOT))
PROJECT_ROOT = BUNDLE_ROOT  # bundle-local numerical source root
OUTPUT_DIR = BUNDLE_ROOT / 'output'
plt.rcParams.update({'axes.labelsize': 11, 'axes.labelweight': 'bold',
                     'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11})
importlib.invalidate_caches()
for module_name in ('dms.config', 'dms.stieltjes', 'dms.simulation', 'dms.theory', 'dms.optimization', 'dms.jobs', 'dms.ridge_diagnostics', 'dms.exploration', 'dms.landscape_slices', 'dms.paper_figures', 'dms.result_cache', 'dms.incremental_simulation', 'dms.transition_diagnostics', 'dms'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

from dms import DMSTTheory, project_stieltjes_evaluator
from dms.exploration import SweepParameters, parameter_sweep_cases
from dms.ridge_diagnostics import optimize_effective_ridge_over_r
from dms.paper_figures import compute_panels
from dms.jobs import gr_job_sizes
from dms.simulation import moment_feasibility
from dms.result_cache import ResultCache
from dms.incremental_simulation import cached_gr_job, STREAM_SCHEME
from dms.transition_diagnostics import ridge_transition_evidence
from matplotlib.colors import Normalize, LogNorm, ListedColormap, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.ticker import NullLocator

theory = DMSTTheory(project_stieltjes_evaluator())
CASES, RESULTS, SIMULATIONS, FIGURES = {}, {}, {}, {}
SIM_METADATA, FIGURE_OPTIONS = {}, {}
TRANSITIONS, TRANSITION_IDS, TRANSITION_SIGNATURES = {}, {}, {}
CACHE_OPTIONS = dict(directory=BUNDLE_ROOT / '.cache' / 'landscapes',
                     enabled=True, verbose=True)

from tools.portable_results import load_results, assert_settings_match

# False: reproduce from the included arrays. True: calculate from the inputs below.
RECOMPUTE = False


## 1. Selected case and numerical controls

The mean calibration length is `L_ref=100`; converted raw moments remain fixed as simulated sizes change. Simulation budget `B` selects `n=max(2,floor(B/sqrt(r)))` and `L=b*floor(B*sqrt(r)/(q*b))`, with `b` the denominator of `eta`, preserving integral mismatch counts. The displayed markers use realized `r=qL/n`.

Fixed `r`/`lambda` slice lists, grids, search cap, seed, and per-case settings are editable here. Map resolution and optimization resolution are separate controls. A finite upper search-cap hit does not establish an infinite optimum.


In [ ]:
# Select exactly one independently reproducible figure; rerun stages below after changing it.
CASE = 'set_1'  # 'set_1': main; 'set_2' / 'set_6': appendix
FORCE_THEORY = False
FORCE_SIMULATION = False
FORCE_REDRAW = False

LN10 = np.log(10.)

COMPUTE_DEFAULTS = dict(
    r_grid=np.unique(np.r_[np.geomspace(LN10*1e-2,LN10*1e2,181),np.linspace(2.5,5.5,101)]),
    lambda_grid=np.geomspace(1e-5,1e2,241),
    fixed_lambdas=[1e-4,1e-3,1e-2,1e-1,1.],
    fixed_rs=[.1,.3,1.,3.,10.],
    search=dict(lambda_max=1e3,n_grid=641),
)

SIM_OPTIONS = dict(run=True, budget=1000, trials=20, seed=17,
                   r_values=np.geomspace(.1,100,20),
                   slice_lambdas=np.geomspace(1e-4,1,9),fixed_r_points=False)

CASES['set_1'] = dict(parameters=SweepParameters(q=20, sigma=1.0, G2=100.0, f=0.1, ftilde=0.2, eta=0.2, F=0.4, FTest=0.4, Fneq=0.2, Cneq=0.03, m=0.02, mneq=0.1, mneqtilde=0.05),
    test_mismatch_mean=0.2,
    reference_L=100,mean_convention='normalized',
    plot=dict(title=""),compute={})

CASES['set_2'] = dict(parameters=SweepParameters(q=20, sigma=1.0, G2=100.0, f=0.1, ftilde=0.2, eta=0.2, F=0.4, FTest=0.5, Fneq=0.2, rhoneq=0.1, m=0.1, mneq=0.2, mneqtilde=0.3),
    test_mismatch_mean=None,
    reference_L=100,mean_convention='normalized',
    plot=dict(title=""),compute={})

CASES['set_6'] = dict(parameters=SweepParameters(q=20, sigma=1.0, G2=100.0, f=0.1, ftilde=0.2, eta=0.2, F=0.4, FTest=0.4, Fneq=0.2, rhoneq=0.25, m=0.05, mneq=-0.1, mneqtilde=0.1),
    test_mismatch_mean=None,
    reference_L=100,mean_convention='normalized',
    plot=dict(title=""),compute={})

CASES["set_1"].setdefault("compute", {}).update(
    fixed_rs=[0.5, 1., 2., 3., 4., 8.],
    fixed_lambdas=[1e-4, 1e-3, 1e-2, 1e-1, 1.],
)

CASES["set_1"].setdefault("simulation", {}).update(run=True)

CASES["set_1"].setdefault("plot", {}).update(title="")

CASES["set_1"].setdefault("plot", {}).setdefault("labels", {}).update(
    fixed_lambda=r"$\lambda = {value_tex}$", fixed_r=r"$r = {value:g}$",
    local_minima="Local minima", map_optimum=r"$\lambda_{\mathrm{opt}}$")

CASES["set_1"]["plot"].update(
    left_xlim=(0.09, 110),
    panel_label_positions=[(-.12, 1.02), (-.12, 1.02), (-.12, 1.02)],
)

CASES["set_1"].setdefault("plot", {}).update(color_limit_mode="risk_ylim")

CASES["set_1"].setdefault("plot", {}).setdefault("legends", {}).setdefault("left", {}).update(
    loc="lower right",  # e.g. "upper left", "upper right", "lower left", "best"
    # bbox_to_anchor=(1., 0.),  # optional fine positioning in axes coordinates
)

CASES["set_2"].setdefault("compute", {}).update(
    fixed_rs=[0.5, 1., 2., 3., 4., 8.],
    fixed_lambdas=[1e-4, 1e-3, 1e-2, 1e-1, 1.],
)

CASES["set_2"].setdefault("simulation", {}).update(run=False)

CASES["set_2"].setdefault("plot", {}).update(title="")

CASES["set_2"].setdefault("plot", {}).setdefault("labels", {}).update(
    fixed_lambda=r"$\lambda = {value_tex}$", fixed_r=r"$r = {value:g}$",
    local_minima="Loc. min.", map_optimum=r"$\lambda_{\mathrm{opt}}$")

CASES["set_2"].setdefault("plot", {}).update(color_limit_mode="risk_ylim")

CASES["set_2"].setdefault("plot", {}).setdefault("legends", {}).setdefault("left", {}).update(
    loc="lower right",  # e.g. "upper left", "upper right", "lower left", "best"
    # bbox_to_anchor=(1., 0.),  # optional fine positioning in axes coordinates
)

CASES["set_6"].setdefault("compute", {}).update(
    fixed_rs=[0.3, 1., 3., 10., 30., 100],
    fixed_lambdas=[1e-4, 1e-3, 1e-2, 1e-1, 1.],
)

CASES["set_6"].setdefault("simulation", {}).update(run=False)

CASES["set_6"].setdefault("plot", {}).update(title="")

CASES["set_6"].setdefault("plot", {}).setdefault("labels", {}).update(
    fixed_lambda=r"$\lambda = {value_tex}$", fixed_r=r"$r = {value:g}$",
    local_minima="Loc. min.", map_optimum=r"$\lambda_{\mathrm{opt}}$")

CASES["set_6"].setdefault("plot", {}).update(color_limit_mode="risk_ylim")

CASES["set_6"].setdefault("plot", {}).setdefault("legends", {}).setdefault("left", {}).update(
    loc="lower right",  # e.g. "upper left", "upper right", "lower left", "best"
    # bbox_to_anchor=(1., 0.),  # optional fine positioning in axes coordinates
)

# The main figure uses 300 trials; the shared fallback above uses 20.
CASES['set_1'].setdefault('simulation', {}).update(trials=300)
if CASE not in CASES:
    raise KeyError(f'Choose one of {list(CASES)}')

# Barrier evidence controls; no classifier based on line steepness or cap status.
TRANSITION_OPTIONS = dict(n_probe=65, barrier_rtol=1e-8)


## 2. Numerical stage helpers

Published arrays are checked against the selected scientific inputs. Recomputed theory, simulation and barrier diagnostics use separate local caches. Plotting and export do not start scientific calculations. Simulation streams use `SeedSequence(seed, spawn_key=(r_index,))`; the requested aspect-ratio order is part of the inputs.


In [ ]:
def merge(base, overrides):
    out=deepcopy(base)
    for k,v in overrides.items():
        out[k]=merge(out[k],v) if isinstance(v,dict) and isinstance(out.get(k),dict) else deepcopy(v)
    return out

def case_config(key):
    spec=CASES[key];base=spec["parameters"]
    mt=spec.get("test_mismatch_mean")
    case,=parameter_sweep_cases(base,"mneqtilde",[base.mneqtilde if mt is None else mt],
        mean_convention=spec["mean_convention"],reference_L=spec["reference_L"])
    return case.config

def controls(key):
    return merge(COMPUTE_DEFAULTS,CASES[key].get("compute",{}))

def simulation_options(key):
    return merge(SIM_OPTIONS,CASES[key].get("simulation",{}))

def encoded(value):
    return json.dumps(value,sort_keys=True,
                      default=lambda x:x.tolist() if isinstance(x,np.ndarray) else str(x))

def signature(key):
    return encoded(dict(config=case_config(key).as_dict(),controls=controls(key)))

def cache():
    return ResultCache(**CACHE_OPTIONS)

def sources(*names):
    return [PROJECT_ROOT / "dms" / (name+".py") for name in names]

THEORY_SOURCES=sources("config","stieltjes","theory","ridge_diagnostics","landscape_slices")
SIM_SOURCES=sources("config","simulation","jobs")

def publication_settings(key):
    return dict(config=case_config(key).as_dict(), controls=controls(key),
                simulation=simulation_plan(key)[1] if simulation_options(key)['run'] else None,
                transition=merge(TRANSITION_OPTIONS, CASES[key].get('transition', {})))


def restore_published_case(key):
    data, metadata = load_results(BUNDLE_ROOT/'data'/f'landscape_{key}', restore_records=True)
    assert_settings_match(metadata['settings'], publication_settings(key))
    RESULTS[key] = dict(data=data['theory'], signature=signature(key), cache_id=data['theory_id'],
                        controls=controls(key), parameters=asdict(CASES[key]['parameters']),
                        reference_L=CASES[key]['reference_L'])
    TRANSITIONS[key] = data['transitions']
    TRANSITION_IDS[key] = data['transition_id']
    TRANSITION_SIGNATURES[key] = encoded(dict(result=data['theory_id'],
        controls=merge(TRANSITION_OPTIONS, CASES[key].get('transition', {}))))
    if data['simulations']:
        SIMULATIONS[key] = data['simulations']
        SIM_METADATA[key] = data['simulation_metadata']
    return data


def ensure_transitions(key, *, compute=False, force=False):
    if not RECOMPUTE:
        restore_published_case(key)
        return TRANSITIONS[key]
    controls=merge(TRANSITION_OPTIONS,CASES[key].get('transition',{}))
    inputs=dict(result=RESULTS[key]['cache_id'],controls=controls)
    sig=encoded(inputs)
    if not force and TRANSITION_SIGNATURES.get(key)==sig:return TRANSITIONS[key]
    data=RESULTS[key]['data']
    value,identity=cache().get('ridge_transitions',inputs,
        lambda:ridge_transition_evidence(data['optimum'],data['config'],**controls),
        sources=sources('transition_diagnostics'),compute=compute,force=force)
    TRANSITIONS[key]=value;TRANSITION_IDS[key]=identity;TRANSITION_SIGNATURES[key]=sig
    return value


def compute_case(key, *, force=False, compute=True):
    """Load or calculate theory only. compute=False never evaluates risks."""
    if not RECOMPUTE:
        return restore_published_case(key)['theory']
    cfg=case_config(key);control=controls(key)
    def produce():
        data=compute_panels(cfg,**control)
        data["slice_optimum"]=optimize_effective_ridge_over_r(
            data["slices"]["fixed_rs"],cfg,**control["search"])
        return data
    data,identity=cache().get("theory",dict(config=cfg.as_dict(),controls=control),produce,
        sources=THEORY_SOURCES,implementation=["panels-v1",inspect.getsource(compute_panels)],
        force=force,compute=compute)
    RESULTS[key]=dict(data=data,signature=signature(key),cache_id=identity,controls=control,
                      parameters=asdict(CASES[key]["parameters"]),reference_L=CASES[key]["reference_L"])
    if compute:ensure_transitions(key,compute=True,force=force)
    return data

def simulation_plan(key):
    cfg=case_config(key);opts=simulation_options(key);control=controls(key)
    common=dict(config=cfg.as_dict(),budget=opts["budget"],trials=opts["trials"])
    plan={"fixed_lambda":dict(common,r_values=np.asarray(opts["r_values"],float),
           ridge_lambdas=np.unique(control["fixed_lambdas"]),seed=opts["seed"])}
    if opts["fixed_r_points"]:
        plan["fixed_r"]=dict(common,r_values=np.unique(control["fixed_rs"]),
            ridge_lambdas=np.asarray(opts["slice_lambdas"],float),seed=opts["seed"]+1)
    # The lambda grid affects only the dotted realized-r theory, not simulations.
    return cfg,plan

def simulation_signature(key):
    cfg,plan=simulation_plan(key)
    return encoded(dict(plan=plan,stream_scheme=STREAM_SCHEME,lambda_grid=controls(key)["lambda_grid"] if "fixed_r" in plan else None))

def simulate_case(key, *, force=False, compute=True):
    """Resume only missing trials; compute=False restores aggregates without sampling."""
    if not RECOMPUTE:
        return restore_published_case(key)['simulations']
    cfg,plan=simulation_plan(key)
    errors=[]
    for kind,inputs in plan.items():
        for nn,ll in gr_job_sizes(inputs["r_values"],cfg,inputs["budget"]):
            report=moment_feasibility(int(ll),cfg)
            if not report.feasible:errors.append(dict(kind=kind,L=int(ll),messages=report.messages))
    if errors:raise ValueError(errors)
    sims={};identities={};reuse_reports={}
    for kind,inputs in plan.items():
        sims[kind],identities[kind],reuse_reports[kind]=cached_gr_job(
            inputs["r_values"],inputs["ridge_lambdas"],cfg,
            inputs["budget"],inputs["trials"],seed=inputs["seed"],
            cache=cache(),force=force,compute=compute)
        job=sims[kind]
        display(dict(case=key,kind=kind,budget=inputs["budget"],trials=inputs["trials"],
                     seed=inputs["seed"],requested_r=job.requested_r,
                     realized_r=job.data[0,:,0],sizes=job.sizes,
                     trial_reuse=reuse_reports[kind]))
    metadata=dict(signature=simulation_signature(key),cache_ids=identities,
                  trial_reuse=reuse_reports)
    if "fixed_r" in sims:
        realized=sims["fixed_r"].data[0,:,0];grid=np.asarray(controls(key)["lambda_grid"])
        curves,identity=cache().get("realized_r_theory",
            dict(config=cfg.as_dict(),r_values=realized,lambda_grid=grid),
            lambda:np.array([[theory.risk(float(lam),float(r),cfg) for lam in grid] for r in realized]),
            sources=THEORY_SOURCES,force=force,compute=compute)
        metadata.update(realized_r_risk=curves,realized_cache_id=identity)
    SIMULATIONS[key]=sims;SIM_METADATA[key]=metadata
    return sims

def plot_case(key, *, force=False, with_simulations=None, plot_overrides=None):
    """Restore cached inputs if needed; never start numerical calculations."""
    if key not in RESULTS or RESULTS[key]["signature"]!=signature(key):
        compute_case(key,compute=False)
    if with_simulations is None:with_simulations=simulation_options(key)["run"]
    sims=None;sim_ids=None;data=dict(RESULTS[key]["data"])
    if with_simulations:
        if key not in SIM_METADATA or SIM_METADATA[key]["signature"]!=simulation_signature(key):
            simulate_case(key,compute=False)
        sims=SIMULATIONS[key];meta=SIM_METADATA[key]
        sim_ids=dict(groups=meta["cache_ids"],realized=meta.get("realized_cache_id"))
        if "realized_r_risk" in meta:data["realized_r_risk"]=meta["realized_r_risk"]
    opts=merge(PLOT_DEFAULTS,CASES[key].get("plot",{}))
    opts=merge(opts,plot_overrides or {})
    if opts['use_transition_diagnostics']:
        data['transition_diagnostics']=ensure_transitions(key,compute=False)

    inputs=dict(theory=RESULTS[key]["cache_id"],transitions=TRANSITION_IDS.get(key),simulations=sim_ids,plot=opts,
                rcparams={k:repr(v) for k,v in plt.rcParams.items()})
    implementation,uncacheable=panel_plot_fingerprint()
    fig,identity=cache().get("figure",inputs,lambda:plot_panels(data,opts,sims),
        implementation=implementation,force=force or uncacheable)
    figure_key=key
    FIGURES[figure_key]=fig
    FIGURE_OPTIONS[figure_key]=dict(case=key,plot=deepcopy(opts),
        theory_cache_id=RESULTS[key]["cache_id"],simulation_cache_ids=sim_ids)
    display(fig);plt.close(fig)
    return fig


## 3. Theoretical risks, optima and barrier evidence

Load the included results, or run the selected case when `RECOMPUTE=True`.


In [ ]:
_ = compute_case(CASE, force=FORCE_THEORY)


## 4. Synthetic simulation

One coefficient landscape with the prescribed moments is generated per size/r and held fixed across trials. Each trial resamples independent-site training sequences and Gaussian label noise, fits ridge regression with an unpenalized intercept, and evaluates **exact population test risk** from site probabilities. There is no sampled test set or validation-set tuning. The five penalties reuse the same training draw and eigendecomposition.

Markers are mean risk; error bars are one standard error `s/sqrt(T)` with sample `ddof=1`, conditional on the generated landscape. They do not include between-landscape variability. Main `set_1`: B=1000, T=300, seed=17; the appendix cases do not display simulations. The seed, ordered inputs and bundled numerical implementation determine the realizations.


In [ ]:
if simulation_options(CASE)['run']:
    _ = simulate_case(CASE, force=FORCE_SIMULATION)
else:
    print(CASE, 'uses theoretical curves only; simulation disabled in this case.')


## 5. Editable renderer
All figure construction remains visible here. It consumes stored arrays only.


In [ ]:
# All figure styling lives in this notebook. Edit these functions freely.
def _artist_options(options, name, *, color=None, overrides=None):
    """Standard Matplotlib keyword arguments; None inherits shared size/colour."""
    kw=dict(options['styles'][name])
    kw.update(overrides or {})
    if kw.get('linewidth', 'absent') is None:kw['linewidth']=options['linewidth']
    if kw.get('markersize', 'absent') is None:kw['markersize']=options['marker_size']
    if color is not None and kw.get('color') is None:kw['color']=color
    return {k:v for k,v in kw.items() if v is not None}


def _lambda_tex(value):
    """Mathtext without dollar signs; preserve ordinary 0 and 1."""
    if value in (0,1):return f'{value:g}'
    if not np.isfinite(value) or value<0:return f'{value:g}'
    exponent=int(np.floor(np.log10(value)))
    mantissa=value/(10.**exponent)
    if np.isclose(mantissa,10.):mantissa=1.;exponent+=1
    if np.isclose(mantissa,1.):return rf'10^{{{exponent}}}'
    return rf'{mantissa:g}\times 10^{{{exponent}}}'


def _curve_label(options, kind, value, display_value):
    labels=options['curve_labels'][kind]
    if value in labels:return labels[value]
    template=options['labels'][kind]
    return None if template is None else template.format(
        value=display_value,value_tex=_lambda_tex(display_value))


def _draw_connector(ax, x0, x1, a, b, base_kw, *, dashed=False, geometry='diagonal', draw_vertical=True):
    """Join sampled endpoints exactly, without shifting to an invented midpoint."""
    if geometry=='diagonal':
        if draw_vertical:ax.plot([x0,x1],[a,b],**dict(base_kw,linestyle='--' if dashed else '-'))
    elif geometry=='anchored':
        # Downward connector starts at the previous sample; upward ends at next.
        anchor=x0 if b<a else x1
        if anchor==x1:ax.plot([x0,x1],[a,a],**dict(base_kw,linestyle='-'))
        if draw_vertical:ax.plot([anchor,anchor],[a,b],**dict(base_kw,linestyle='--' if dashed else '-'))
        if anchor==x0:ax.plot([x0,x1],[b,b],**dict(base_kw,linestyle='-'))
    else:raise ValueError('connector_geometry must be diagonal or anchored')


def _optimum_path(ax,r,y,options,boundary=None,jump_mask=None):
    style=options['jump_style'];positions=options.get('jump_positions',[])
    if style not in ('solid','dashed','gap'):raise ValueError('Invalid jump_style')
    edge=np.zeros(len(y),int) if boundary is None else np.asarray(boundary)
    mask=np.zeros(len(y)-1,bool) if jump_mask is None else np.asarray(jump_mask,bool)
    kw=_artist_options(options,'map_optimum')
    plain=dict(kw,marker='None');border=dict(plain,linestyle='-',clip_on=False)
    for i in range(len(r)-1):
        a,b=y[i:i+2]
        if not np.isfinite(a+b) or min(a,b)<=0:continue
        marked=mask[i] or any(r[i]<value<=r[i+1] for value in positions)
        if edge[i]!=0 or edge[i+1]!=0 or marked:
            _draw_connector(ax,r[i],r[i+1],a,b,border,
                dashed=marked and style=='dashed',geometry=options['connector_geometry'],
                draw_vertical=not marked or style!='gap')
        else:ax.plot(r[i:i+2],[a,b],**plain)
    good=np.isfinite(y)&(y>0)
    ax.plot(np.asarray(r)[good],np.asarray(y)[good],**dict(kw,linestyle='None',clip_on=False))
    ax.plot([],[],label=options['labels']['map_optimum'],**kw)


def _shared_panel_axes(o):
    """Visual order: map, risk strip, fixed ridge, fixed aspect ratio."""
    layout=o['layout'];widths=layout['panel_widths']
    if len(widths)!=3 or min(widths)<=0:raise ValueError('panel_widths needs three positive entries')
    ratios=[widths[0],layout['outer_gap'],layout['bar_width'],
            layout['axis_gap'],widths[1],layout['outer_gap'],widths[2]]
    if min(ratios)<=0:raise ValueError('Layout widths/gaps must be positive')
    fig=plt.figure(figsize=(o['width'],o['height']),dpi=o['dpi'],layout='constrained')
    fig.get_layout_engine().set(w_pad=layout['constrained_w_pad'],
        h_pad=layout['constrained_h_pad'],wspace=layout['constrained_wspace'],
        hspace=layout['constrained_hspace'])
    gs=fig.add_gridspec(1,7,width_ratios=ratios,wspace=0)
    # Keep logical names for backwards-compatible per-panel options:
    # left = fixed ridge (now centre); mid = map (now first).
    mid=fig.add_subplot(gs[0,0]);left=fig.add_subplot(gs[0,4])
    right=fig.add_subplot(gs[0,6],sharey=left)
    risk_bar=fig.add_subplot(gs[0,2],sharey=left)
    mid.set_yscale('log')
    return fig,(left,mid,right),risk_bar


def _lambda_encoding(o, values, limits):
    if len(limits)!=2 or not 0<limits[0]<limits[1]:raise ValueError('Positive increasing ridge limits required')
    lo,hi=o['lambda_color_span']
    if not 0<=lo<hi<=1:raise ValueError('lambda_color_span must increase within [0,1]')
    cmap=ListedColormap(plt.get_cmap(o['lambda_cmap'])(np.linspace(lo,hi,512)))
    # Spread sampled lambdas across the selected palette span, independently of map limits.
    low,high=float(np.min(values)),float(np.max(values))
    if low==high:low,high=low/10,high*10
    norm=LogNorm(low,high,clip=True)
    return cmap,norm,cmap(norm(values))


def _draw_risk_strip(ax, limits, cmap, norm, o):
    """Continuous scale on exactly the right panel's risk coordinates."""
    low,high=limits
    edges=np.linspace(low,high,o['layout']['bar_resolution']+1)
    ax.pcolormesh([0,1],edges,((edges[:-1]+edges[1:])/2)[:,None],
                  cmap=cmap,norm=norm,shading='flat',rasterized=True)
    ax.set_xlim(0,1);ax.set_ylim(low,high)
    if o['layout']['risk_extensions']:
        size=o['layout']['extension_size']
        for vertices,value in [([(0,0),(1,0),(.5,-size)],low),
                               ([(0,1),(1,1),(.5,1+size)],high)]:
            ax.add_patch(Polygon(vertices,transform=ax.transAxes,facecolor=cmap(norm(value)),
                                 edgecolor=o['layout']['bar_edgecolor'],clip_on=False,
                                 linewidth=o['layout']['bar_linewidth']))


def _risk_background(ax, levels, cmap, norm, o):
    """Horizontal bands encode the ordinate's risk, not an extra calculation."""
    opts=o['risk_background'];low,high=ax.get_ylim()
    if opts['mode']=='continuous':
        edges=np.linspace(low,high,opts['resolution']+1)
        values=(edges[:-1]+edges[1:])/2
    elif opts['mode']=='bands':
        edges=np.unique(np.r_[low,levels[(levels>low)&(levels<high)],high])
        middle=(edges[:-1]+edges[1:])/2
        ids=np.searchsorted(levels,middle,side='right')-1
        clipped=np.clip(ids,0,len(levels)-2)
        values=(levels[clipped]+levels[clipped+1])/2
        values=np.where(ids<0,norm.vmin-1,values)
        values=np.where(ids>=len(levels)-1,norm.vmax+1,values)
    else:raise ValueError('risk_background mode must be bands or continuous')
    ax.pcolormesh([0,1],edges,values[:,None],transform=ax.get_yaxis_transform(),
                  cmap=cmap,norm=norm,shading='flat',alpha=opts['alpha'],
                  zorder=opts['zorder'],rasterized=True)
    ax.set_ylim(low,high)  # Background must not expand the data limits.


def _sweep_legend(ax,o,kind,values,colors,factor,legend_options):
    """Only swept values: solid colour swatches, no theory/numerical entries."""
    handles=[]
    for value,color in zip(values,colors):
        kw=_artist_options(o,kind,color=color,overrides=o['curve_styles'][kind].get(value))
        kw.update(linestyle='-',marker='None')
        label=_curve_label(o,kind,value,value*factor)
        if label is not None:
            handles.append(Line2D([],[],label=label,**kw))
    legend_options=dict(legend_options)
    if legend_options.get('title') is None:legend_options['title']=''
    if handles:ax.legend(handles=handles,**legend_options)


def _finish_shared_scales(axes,bars,values,colors,lambda_cmap,lambda_norm,risk_norm,levels,o):
    left,mid,right=axes;risk_bar=bars;layout=o['layout']
    if layout['label_mode'] not in ('axis','above_colorbar'):
        raise ValueError('label_mode must be axis or above_colorbar')
    display_limits=o['map_ylim'] or mid.get_ylim()
    mid.set_ylim(display_limits)
    # Fixed-ridge panel owns the risk limits; no independent right_ylim needed.
    risk_limits=o['left_ylim'] if o['left_ylim'] is not None else left.get_ylim()
    left.set_ylim(risk_limits)  # Propagates to both risk panels and the strip.
    risk_cmap=plt.get_cmap(o['risk_cmap'])
    _draw_risk_strip(risk_bar,risk_limits,risk_cmap,risk_norm,o)
    if layout['lambda_ticks'] is not None:
        ticks=np.asarray(layout['lambda_ticks'],float)
        mid.set_yticks(ticks[(ticks>=display_limits[0])&(ticks<=display_limits[1])])
        mid.set_ylim(display_limits)
    if o['colorbar_ticks'] is not None:
        ticks=np.asarray(o['colorbar_ticks'],float)
        left.set_yticks(ticks[(ticks>=risk_limits[0])&(ticks<=risk_limits[1])])
    if not layout['risk_minor_ticks']:left.yaxis.set_minor_locator(NullLocator())
    for panel,bar in ((left,risk_bar),):
        # Do not clear locators on a shared axis: only hide duplicate artists.
        bar.set_xticks([])
        bar.tick_params(axis='y',which='both',left=False,right=False,labelleft=False,labelright=False)
        panel.tick_params(axis='y',which='both',left=True,right=False,labelleft=True,labelright=False)
        for spine in bar.spines.values():
            spine.set_color(layout['bar_edgecolor']);spine.set_linewidth(layout['bar_linewidth'])
        label=panel.get_ylabel()
        if bar is risk_bar and o['colorbar_label'] is not None:label=o['colorbar_label']
        if layout['label_mode']=='above_colorbar':
            panel.set_ylabel('')
            bar.set_title(label,fontsize=o['label_size'],pad=layout['bar_title_pad'])
        else:panel.set_ylabel(label,fontsize=o['label_size'])
    right.set_ylabel('')
    right.tick_params(axis='y',which='both',left=False,right=False,labelleft=False,labelright=False)
    bg=o['risk_background']
    if not 0<=bg['alpha']<=1:raise ValueError('risk_background alpha must lie in [0,1]')
    if bg['enabled']:
        if bg['left']:_risk_background(left,levels,risk_cmap,risk_norm,o)
        if bg['right']:_risk_background(right,levels,risk_cmap,risk_norm,o)
    # Keep the data panels above their gradient backgrounds.
    for ax in (left,right):ax.set_axisbelow(True)


def plot_panels(data, options, simulations=None):
    """Editable Figure 1/3 plotting only: no theory or simulation calls."""
    o=options;sl=data['slices'];cfg=data['config'];opt=data['optimum']
    if o['coordinate'] not in ('lambda','effective'):raise ValueError('Unknown ridge coordinate')
    factor=1 if o['coordinate']=='lambda' else (cfg.q-1)/cfg.f_train
    ridge_label=o['ridge_label'] or (r'Ridge $\lambda$' if o['coordinate']=='lambda' else r'Effective $\lambda$')
    scale=sl['R_infinity'] if o['normalize'] else 1.
    ylabel=o['risk_label'] or (r'$R/R_\infty$' if o['normalize'] else r'$R$')
    r,y,z=data['r'],data['ridge']*factor,data['risk']/scale
    fig,axes,bars=_shared_panel_axes(o)
    left,mid,right=axes
    lambda_values=np.asarray(sl['fixed_lambdas'])*factor
    lambda_limits=o['map_ylim'] or (y[0],y[-1])
    lambda_cmap,lambda_norm,lc=_lambda_encoding(o,lambda_values,lambda_limits)
    # Legend swatches and simulation markers follow explicit curve colour overrides.
    lc=np.asarray([to_rgba(_artist_options(o,'fixed_lambda',color=color,
        overrides=o['curve_styles']['fixed_lambda'].get(lam))['color'])
        for lam,color in zip(sl['fixed_lambdas'],lc)])
    rc=plt.get_cmap(o['r_cmap'])(np.linspace(*o['r_color_span'],len(sl['fixed_rs'])))
    for lam,risk,col in zip(sl['fixed_lambdas'],sl['fixed_lambda_risk'],lc):
        kw=_artist_options(o,'fixed_lambda',color=col,overrides=o['curve_styles']['fixed_lambda'].get(lam))
        left.plot(r,risk/scale,label=_curve_label(o,'fixed_lambda',lam,lam*factor),**kw)
    if o['show_optimum'] and o['show_optimal_risk']:
        left.plot(r,opt.risks/scale,label=o['labels']['optimized_risk'],**_artist_options(o,'optimized_risk'))
    if simulations and 'fixed_lambda' in simulations:
        job=simulations['fixed_lambda']
        if job.config!=cfg:raise ValueError('Simulation/config mismatch')
        for i,(lam,col) in enumerate(zip(sl['fixed_lambdas'],lc)):
            indices=np.flatnonzero(np.isclose(job.ridge_lambdas,lam,rtol=1e-12,atol=0))
            if len(indices)!=1:raise ValueError('Simulation penalty list changed; recompute')
            points=job.data[indices[0]]
            kw=_artist_options(o,'simulation',color=col)
            kw.update(o['simulation_styles']['fixed_lambda'].get(lam,{}))
            left.errorbar(points[:,0],points[:,1]/scale,yerr=points[:,2]/scale,
                          label=o['labels']['simulation'] if i==0 else None,**kw)
    for rv,risk,col in zip(sl['fixed_rs'],sl['fixed_r_risk'],rc):
        kw=_artist_options(o,'fixed_r',color=col,overrides=o['curve_styles']['fixed_r'].get(rv))
        right.plot(y,risk/scale,label=_curve_label(o,'fixed_r',rv,rv),**kw)
    if simulations and 'fixed_r' in simulations:
        job=simulations['fixed_r']
        if job.config!=cfg:raise ValueError('Simulation/config mismatch')
        for i,(rv,col) in enumerate(zip(sl['fixed_rs'],rc)):
            ids=np.flatnonzero(np.isclose(job.requested_r,rv,rtol=1e-12,atol=0))
            if len(ids)!=1:raise ValueError('Simulation r list changed; recompute')
            j=ids[0];pts=job.data[:,j]
            kw=_artist_options(o,'simulation',color=col)
            kw.update(o['simulation_styles']['fixed_r'].get(rv,{}))
            right.errorbar(job.ridge_lambdas*factor,pts[:,1]/scale,yerr=pts[:,2]/scale,
                           label=o['labels']['simulation'] if i==0 else None,**kw)
            if o['show_realized_r_theory'] and 'realized_r_risk' in data:
                right.plot(y,data['realized_r_risk'][j]/scale,
                           label=o['labels']['realized_r'] if i==0 else None,**_artist_options(o,'realized_r',color=col))
    if o['parametric_optima']:
        ok=(opt.ridge_lambdas>0)&(opt.status!='upper')
        right.plot(opt.ridge_lambdas[ok]*factor,opt.risks[ok]/scale,
                   label=o['labels']['locus'],**_artist_options(o,'locus'))
    if o['show_reference']:
        for ax in (left,right):ax.axhline(sl['R_infinity']/scale,label=o['labels']['reference'],**_artist_options(o,'reference'))
    # Both risk panels have been drawn, so automatic shared limits are resolved.
    mode=o['color_limit_mode']
    if mode=='fixed':limits=(0.,1.)
    elif mode=='risk_ylim':limits=o['left_ylim'] if o['left_ylim'] is not None else left.get_ylim()
    elif mode=='custom':limits=o['color_limits']
    else:raise ValueError('color_limit_mode must be fixed, risk_ylim, or custom')
    limits=np.asarray(limits,float)
    if limits.shape!=(2,) or not np.all(np.isfinite(limits)) or limits[0]>=limits[1]:
        raise ValueError('Palette limits must be two finite increasing values')
    norm=Normalize(*limits)
    levels=o['contour_levels']
    if levels is None:levels=np.linspace(*limits,o['contour_count'])
    levels=np.asarray(levels,float)
    if len(levels)<2 or np.any(np.diff(levels)<=0):raise ValueError('Contour levels must increase')
    mid.contourf(r,y,z,levels=levels,cmap=o['risk_cmap'],norm=norm,extend='both')
    if o['show_contour_lines']:mid.contour(r,y,z,levels=levels,**o['contour_style'])
    if o['show_local_minima']:
        ii,jj=np.nonzero((z[1:-1]<z[:-2])&(z[1:-1]<z[2:]))
        mid.plot(r[jj],y[ii+1],label=o['labels']['local_minima'],**_artist_options(o,'local_minima'))
    if o['show_optimum']:
        best=np.asarray(opt.ridge_lambdas*factor,float)
        bottom,top=o['map_ylim'] or (y[0],y[-1])
        boundary=np.zeros(len(best),int)
        boundary[(best<=bottom)|(opt.status=='zero')]=-1
        boundary[(best>=top)|np.isposinf(best)|(opt.status=='upper')]=1
        plotted=np.clip(best,bottom,top)
        plotted[boundary==-1]=bottom;plotted[boundary==1]=top
        plotted[np.isnan(best)|(best<0)]=np.nan
        # Display projection only: stored optima/status/risk are untouched.
        evidence=data.get('transition_diagnostics') if o['use_transition_diagnostics'] else None
        mask=evidence['jump_mask'] if evidence is not None else None
        _optimum_path(mid,r,plotted,o,boundary=boundary,jump_mask=mask)
        zero=opt.status=='zero'
        if o['show_zero_strip'] and np.any(zero):
            mid.plot(r,np.where(zero,o['zero_strip_y'],np.nan),transform=mid.get_xaxis_transform(),
                     **_artist_options(o,'zero'))
            mid.text(*o['zero_label_position'],o['labels']['zero'],transform=mid.transAxes,
                     fontsize=o['legend_size'],**o['zero_text_style'])
    left.set_xscale('log');mid.set_xscale('log');mid.set_yscale('log');right.set_xscale('log')
    for ax,name,xlabel,ylab,title in zip(axes,('left','map','right'),
            (o['r_label'],o['r_label'],ridge_label),(ylabel,ridge_label,ylabel),o['panel_titles']):
        ax.set_xlabel(o['axis_labels'].get(name+'_x',xlabel),fontsize=o['label_size'])
        ax.set_ylabel(o['axis_labels'].get(name+'_y',ylab),fontsize=o['label_size'])
        ax.tick_params(labelsize=o['tick_size'])
        if o[name+'_xlim'] is not None:ax.set_xlim(o[name+'_xlim'])
        if name!='right' and o[name+'_ylim'] is not None:ax.set_ylim(o[name+'_ylim'])
        if name=='right' and o['mark_slice_optima'] and data.get('slice_optimum') is not None:
            # Position after limits are resolved: markers must not expand the axis.
            so=data['slice_optimum'];xlo,xhi=ax.get_xlim();saved_ylim=ax.get_ylim()
            for i,(lam,risk,status) in enumerate(zip(so.ridge_lambdas,so.risks,so.status)):
                if np.isnan(lam) or lam<0 or not np.isfinite(risk):continue
                value=lam*factor
                upper=(status=='upper') or np.isposinf(value) or value>xhi
                lower=(status=='zero') or value<xlo
                inside=not (upper or lower)
                if not inside and not o['pin_outside_slice_optima']:continue
                if upper:xpos=1.
                elif lower:xpos=0.
                else:xpos=(np.log(value)-np.log(xlo))/(np.log(xhi)-np.log(xlo))
                kw=_artist_options(o,'slice_optima',color=rc[i])
                kw.update(clip_on=False,label='_nolegend_')
                ax.plot([xpos],[risk/scale],transform=ax.get_yaxis_transform(),**kw)
            ax.set_xlim(xlo,xhi);ax.set_ylim(saved_ylim)
        ax.set_title(title,fontsize=o['title_size'])
        legend=dict(o['legends'][name]);show=legend.pop('show')
        legend.setdefault('fontsize',o['legend_size']);legend.setdefault('title_fontsize',o['legend_size'])
        if o['legend'] and show:
            if name=='left':
                _sweep_legend(ax,o,'fixed_lambda',sl['fixed_lambdas'],lc,factor,legend)
            elif name=='right':
                _sweep_legend(ax,o,'fixed_r',sl['fixed_rs'],rc,1.,legend)
            else:ax.legend(**legend)
    _finish_shared_scales(axes,bars,lambda_values,lc,lambda_cmap,lambda_norm,norm,levels,o)
    # Panel letters follow visual order, excluding the colourbar.
    for ax, label, position in zip((mid, left, right),
            o.get('panel_labels', ('a', 'b', 'c')),
            o.get('panel_label_positions', [(-.12, 1.02)] * 3)):
        style = dict(fontsize=12, fontweight='bold', ha='left', va='bottom', clip_on=False)
        style.update(o.get('panel_label_style', {}))
        ax.text(*position, label, transform=ax.transAxes, **style)
    fig.suptitle(o['title'],fontsize=o['title_size'])
    return fig


def panel_plot_fingerprint():
    """Fingerprint the live notebook functions; never serve an old-code figure."""
    functions=(_draw_connector,_artist_options,_lambda_tex,_curve_label,_optimum_path,_shared_panel_axes,
               _lambda_encoding,_draw_risk_strip,_risk_background,
               _sweep_legend,_finish_shared_scales,plot_panels)
    try:
        return [inspect.getsource(fn) for fn in functions],False
    except (OSError,TypeError):
        # If this kernel cannot retrieve cell source, redraw rather than reuse.
        return [],True


## 6. Presentation controls

Visual order is map, fixed-lambda cuts, fixed-r cuts. Legacy `left` names mean the central panel; its legend defaults to lower right. Fonts, markers, limits, contours, colours, and independent legend positions are editable below. `color_limit_mode` changes colours only. Dashed connectors use stored barrier/manual evidence and join sampled endpoints.


In [ ]:
PLOT_DEFAULTS = dict(
    panel_labels=('a', 'b', 'c'),  # visual left-to-right order; () hides labels
    panel_label_positions=[(-.12, 1.02),(-.12, 1.02),(-.02, 1.02)] ,  # axes coordinates, independently editable
    panel_label_style=dict(fontsize=12, fontweight='bold'),
    width=10., height=3.4, dpi=140,
    coordinate="lambda", normalize=True,
    label_size=11, tick_size=11, title_size=12, legend_size=11,
    linewidth=1.8, marker_size=2,
    lambda_cmap="magma", r_cmap="viridis", risk_cmap="turbo",
    color_limit_mode="fixed",  # "fixed": 0–1; "risk_ylim": shared risk y-limits; "custom": color_limits.
    contour_count=30, contour_levels=None, color_limits=(0,1.05), colorbar_ticks=[0,1],
    left_xlim=(LN10*1e-2,LN10*1e2), left_ylim=(-.03,1.03),
    map_xlim=(LN10*1e-2,LN10*1e2), map_ylim=(LN10*1e-5,LN10),
    right_xlim=None, right_ylim=None,
    r_label=r"Aspect ratio $r=qL/n$", title="",
    panel_titles=["","",""], #["Fixed ridge", "Risk landscape", "Fixed aspect ratio"],
    show_optimum=True, show_optimal_risk=False, show_local_minima=True, mark_slice_optima=True,
    parametric_optima=False, jump_style="dashed", jump_ratio=5., legend=True,
)

# Detailed plot controls. Standard Matplotlib keyword names are used below.
# linewidth=None / markersize=None inherit the shared linewidth / marker_size.
# marker='None' hides markers; linestyle='None' hides connecting lines.
# markerfacecolor='none' makes hollow markers. Sizes are in points.
PLOT_DEFAULTS.update(
    jump_positions=[],  # Only explicitly identified transition r values get dashed connectors.
    ridge_label=None, risk_label=None, colorbar_label=None,  # None = automatic label
    axis_labels={"left_x": r"Aspect ratio $r = qL/n$", "map_x": r"Aspect ratio $r = qL/n$"},  # e.g. {'left_y': r'$R/R_\infty$', 'right_x': 'Ridge'}
    lambda_color_span=(.15,.85),  # Broad magma range; omit near-black and pale yellow.
    r_color_span=(.05,.9),
    pin_outside_slice_optima=True,  # Off-range minima go to the corresponding left/right edge.
    show_contour_lines=True, show_reference=True, show_realized_r_theory=True,
    contour_style=dict(colors='black',linewidths=.5,linestyles='solid',alpha=.45),
    show_zero_strip=False,  # Hide the zero-ridge strip and label for now.
    zero_strip_y=-.09, zero_label_position=(0,-.14),
    zero_text_style=dict(rotation=0,ha='left',va='top'),
    styles=dict(
        fixed_lambda=dict(linestyle='-',linewidth=None,marker='None',markersize=None),
        fixed_r=dict(linestyle='-',linewidth=None,marker='None',markersize=None),
        optimized_risk=dict(color='black',linestyle='--',linewidth=None,marker='None',markersize=None),
        simulation=dict(marker='o',markersize=None,markerfacecolor=None,markeredgecolor=None,
                        markeredgewidth=1.,linestyle='None',capsize=2,elinewidth=None,capthick=None,alpha=1.),
        local_minima=dict(marker='o',markersize=4,markerfacecolor='none',markeredgewidth=.8,
                          color='0.6',linestyle='None',alpha=1.,zorder=3),
        map_optimum=dict(color='black',linestyle='-',linewidth=1.5,marker='None',markersize=None,zorder=4),
        jump=dict(color=None,linestyle='--',linewidth=None,marker='None',zorder=4),
        slice_optima=dict(marker='o',markersize=4,markerfacecolor=None,
                          markeredgewidth=1.,linestyle='None',zorder=5),
        upper=dict(color='black',marker='^',markersize=4,markerfacecolor='none',
                   linestyle='None',clip_on=False,zorder=5),
        zero=dict(color='black',linestyle='-',linewidth=None,marker='None',markersize=None,clip_on=False),
        realized_r=dict(linestyle=':',linewidth=1.,marker='None',markersize=None),
        locus=dict(color='black',marker='o',markersize=np.sqrt(5),linestyle='None',alpha=.4),
        reference=dict(color='0.6',linestyle=':',linewidth=1.,marker='None'),
    ),
    # Any entry may be changed per case. None hides an optional legend entry.
    # {value} is numeric; {value_tex} formats lambda as a power of ten.
    labels=dict(fixed_lambda=r'$\lambda={value_tex}$',fixed_r=r'$r={value:g}$',
                optimized_risk=r'$R_{\mathrm{opt}}$',
                local_minima='Local minima',map_optimum=r'Optimal $\lambda_{\mathrm{opt}}$',
                upper='Search cap hit',zero=r'$\lambda_{opt}=0$',
                slice_optima=None,locus='Optimal-point locus',
                simulation=None,realized_r=None,reference=None),
    # Keys are actual ridge lambda / r values, independent of display coordinate.
    curve_labels=dict(fixed_lambda={},fixed_r={}),
    curve_styles=dict(fixed_lambda={},fixed_r={}),
    simulation_styles=dict(fixed_lambda={},fixed_r={}),
    # title=None uses the default; title='' hides it. Set show=False per panel.
    # Other legend kwargs can be added: bbox_to_anchor, ncol, markerscale, etc.
    # 'left' is the central fixed-lambda panel (b), not the map.
    legends=dict(left=dict(show=True,title='',loc='lower right',frameon=True),
                 map=dict(show=True,title=None,loc='best',frameon=True),
                 right=dict(show=True,title='',loc='best',frameon=True)),
)

# Risk-barrier evidence is computed/cacheable separately from the optimizer.
TRANSITION_OPTIONS=dict(n_probe=65,barrier_rtol=1e-8)
PLOT_DEFAULTS.update(connector_geometry="diagonal",use_transition_diagnostics=True)
# connector_geometry="anchored": down at previous sample, up at next sample.


PLOT_DEFAULTS.update(
    layout=dict(
        label_mode="axis",              # "axis" or "above_colorbar"
        panel_widths=(1.,1.,1.), bar_width=.055,
        outer_gap=.01, axis_gap=.025,     # relative spacer widths, not data coordinates
        constrained_w_pad=.01, constrained_h_pad=.03,  # inches of automatic padding
        constrained_wspace=0., constrained_hspace=0.,  # fractional layout padding
        lambda_ticks=None,              # None: automatic logarithmic ticks
        risk_minor_ticks=False,         # risk major ticks use colorbar_ticks (default [0,1])
        bar_resolution=512, bar_edgecolor="0.25", bar_linewidth=.7,
        risk_extensions=True, extension_size=.025, bar_title_pad=8,
    ),
    risk_background=dict(enabled=False,left=True,right=True,
                         mode="bands",alpha=.4,resolution=512,zorder=0),
)

PLOT_DEFAULTS["labels"].update(theory=None,simulation=None)


In [ ]:
# Final, plot-only overrides beside the plot call.
RISK_LIMITS = {'set_1': (-.03, 1.33), 'set_2': (-.03, 1.83), 'set_6': (-.03, 1.03)}
CASE_PLOT_OVERRIDES = dict(left_ylim=RISK_LIMITS[CASE])
# Example: CASE_PLOT_OVERRIDES['legends'] = {'left': {'loc': 'lower right'}}
_ = plot_case(CASE, force=FORCE_REDRAW, plot_overrides=CASE_PLOT_OVERRIDES)


## 7. Save the displayed PDF
This cell saves existing figures only; it does not evaluate risks or start trials.


In [ ]:
# Save only figures already displayed by the plot cell. No calculation or plotting occurs here.
PDF_FILENAMES = {
    'set_1': 'shared_axes_set_1.pdf',
    'set_2': 'appendix_landscape_set_2.pdf',
    'set_6': 'appendix_landscape_set_6.pdf',
}
PDF_FIGURES = {CASE: PDF_FILENAMES[CASE]}
missing = [name for name in PDF_FIGURES if name not in FIGURES]
if missing:
    raise RuntimeError(f'Display these figures before export: {missing}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, filename in PDF_FIGURES.items():
    FIGURES[name].savefig(OUTPUT_DIR / filename, format='pdf', bbox_inches='tight')
    print(OUTPUT_DIR / filename)
